# Stable Identity Slice

Purpose: prove the first stable-identity slice over promoted entities, including explicit alias membership and explicit external-reference state.

Mode: proof

Related docs:
- `docs/plans/0001_successor_roadmap.md`
- `docs/plans/0007_phase12_identity_shape.md`
- `docs/adr/0013-start-stable-identity-with-promoted-entity-identities-alias-membership-and-explicit-external-reference-state.md`

## Phase 1

Purpose: seed two promoted entities that should later be grouped under one stable identity.

Input -> output: `review_seed_inputs -> promoted_entity_seed`

Acceptance criteria:
1. Two entity-bearing candidates are accepted and promoted.
2. The promoted graph contains two distinct promoted entity ids.
3. The proof uses real services and explicit state transitions.

status: `proven`

execution_mode: `live`

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Could not locate onto-canon6 repo root from notebook cwd")

for candidate_path in (PROJECT_ROOT / "src", PROJECT_ROOT.parent / "llm_client"):
    if candidate_path.exists():
        candidate_str = str(candidate_path)
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)

import contextlib
import io
import json
from tempfile import TemporaryDirectory

from onto_canon6 import cli as cli_module
from onto_canon6.core import CanonicalGraphService
from onto_canon6.pipeline import ReviewService


In [2]:
workspace = TemporaryDirectory()
workspace_path = Path(workspace.name)
review_db_path = workspace_path / 'review.sqlite3'
overlay_root = workspace_path / 'ontology_overlays'

def run_cli(args: list[str]) -> tuple[int, str, str]:
    stdout_buffer = io.StringIO()
    stderr_buffer = io.StringIO()
    with contextlib.redirect_stdout(stdout_buffer), contextlib.redirect_stderr(stderr_buffer):
        exit_code = cli_module.main(args)
    return exit_code, stdout_buffer.getvalue(), stderr_buffer.getvalue()

review_service = ReviewService(
    db_path=review_db_path,
    overlay_root=overlay_root,
    default_acceptance_policy='record_only',
)

def submit_and_promote(predicate: str, entity_id: str, source_ref: str) -> str:
    submission = review_service.submit_candidate_assertion(
        payload={
            'predicate': predicate,
            'roles': {
                'subject': [
                    {'entity_id': entity_id, 'entity_type': 'oc:person'}
                ],
                'descriptor': [
                    {'kind': 'value', 'value_kind': 'string', 'value': 'identity notebook demo'}
                ],
            },
        },
        profile_id='default',
        profile_version='1.0.0',
        submitted_by='analyst:notebook',
        source_kind='text_file',
        source_ref=source_ref,
        source_text='Identity notebook source text.',
    )
    accepted = review_service.review_candidate(
        candidate_id=submission.candidate.candidate_id,
        decision='accepted',
        actor_id='analyst:reviewer',
    )
    CanonicalGraphService(db_path=review_db_path).promote_candidate(
        candidate_id=accepted.candidate_id,
        promoted_by='analyst:graph-promoter',
    )
    return accepted.candidate_id

candidate_id_one = submit_and_promote('oc:identity_notebook_one', 'ent:person:eric_olson', 'notes/identity_one.txt')
candidate_id_two = submit_and_promote('oc:identity_notebook_two', 'ent:person:admiral_eric_olson', 'notes/identity_two.txt')
seed_summary = {
    'candidate_ids': [candidate_id_one, candidate_id_two],
    'entity_ids': ['ent:person:eric_olson', 'ent:person:admiral_eric_olson'],
}
seed_summary

{'candidate_ids': ['cand_e8151263664444a5a441dfe8',
  'cand_3545e3570a5243f9ba527b3f'],
 'entity_ids': ['ent:person:eric_olson', 'ent:person:admiral_eric_olson']}

## Phase 2

Purpose: create the stable identity and attach explicit alias membership.

Input -> output: `promoted_entity_seed -> identity_membership_state`

Acceptance criteria:
1. The same identity is reused for repeated creation of the canonical entity.
2. Alias membership is explicit and auditable.
3. Identity state is inspectable through the CLI.

status: `proven`

execution_mode: `live`

In [3]:
create_exit, create_stdout, create_stderr = run_cli([
    'create-identity-for-entity',
    '--entity-id', 'ent:person:eric_olson',
    '--actor-id', 'analyst:identity',
    '--display-label', 'Eric Olson',
    '--review-db-path', str(review_db_path),
    '--output', 'json',
])
assert create_exit == 0, create_stderr
identity_bundle = json.loads(create_stdout)
identity_id = identity_bundle['identity']['identity_id']

repeat_exit, repeat_stdout, repeat_stderr = run_cli([
    'create-identity-for-entity',
    '--entity-id', 'ent:person:eric_olson',
    '--actor-id', 'analyst:identity',
    '--display-label', 'Eric Olson',
    '--review-db-path', str(review_db_path),
    '--output', 'json',
])
assert repeat_exit == 0, repeat_stderr
repeat_bundle = json.loads(repeat_stdout)
assert repeat_bundle['identity']['identity_id'] == identity_id

alias_exit, alias_stdout, alias_stderr = run_cli([
    'attach-identity-alias',
    '--identity-id', identity_id,
    '--entity-id', 'ent:person:admiral_eric_olson',
    '--actor-id', 'analyst:identity',
    '--review-db-path', str(review_db_path),
    '--output', 'json',
])
assert alias_exit == 0, alias_stderr
alias_membership = json.loads(alias_stdout)
identity_membership_state = {
    'identity_bundle': identity_bundle,
    'alias_membership': alias_membership,
}
identity_membership_state

{'identity_bundle': {'external_references': [],
  'identity': {'created_at': '2026-03-18T19:40:48.468553+00:00',
   'created_by': 'analyst:identity',
   'display_label': 'Eric Olson',
   'identity_id': 'gid_d9199a02acfe3e5ab6c8a0b1',
   'identity_kind': 'entity'},
  'memberships': [{'attached_at': '2026-03-18T19:40:48.468637+00:00',
    'attached_by': 'analyst:identity',
    'entity_id': 'ent:person:eric_olson',
    'identity_id': 'gid_d9199a02acfe3e5ab6c8a0b1',
    'membership_kind': 'canonical'}],
  'promoted_entities': [{'created_at': '2026-03-18T19:40:48.417305+00:00',
    'entity_id': 'ent:person:eric_olson',
    'entity_type': 'oc:person',
    'first_candidate_id': 'cand_e8151263664444a5a441dfe8'}]},
 'alias_membership': {'attached_at': '2026-03-18T19:40:48.478788+00:00',
  'attached_by': 'analyst:identity',
  'entity_id': 'ent:person:admiral_eric_olson',
  'identity_id': 'gid_d9199a02acfe3e5ab6c8a0b1',
  'membership_kind': 'alias'}}

## Phase 3

Purpose: persist explicit attached and unresolved external-reference state and export the identity report.

Input -> output: `identity_membership_state -> identity_report`

Acceptance criteria:
1. External references are explicit attached or unresolved records.
2. The identity report exposes memberships, promoted entities, and external refs together.
3. The output remains a downstream JSON artifact.

status: `proven`

execution_mode: `live`

In [4]:
attached_exit, attached_stdout, attached_stderr = run_cli([
    'attach-external-ref',
    '--identity-id', identity_id,
    '--provider', 'wikidata',
    '--external-id', 'Q5388397',
    '--reference-label', 'Eric Olson',
    '--actor-id', 'analyst:identity',
    '--review-db-path', str(review_db_path),
    '--output', 'json',
])
assert attached_exit == 0, attached_stderr
attached_reference = json.loads(attached_stdout)

unresolved_exit, unresolved_stdout, unresolved_stderr = run_cli([
    'record-unresolved-external-ref',
    '--identity-id', identity_id,
    '--provider', 'wikidata',
    '--unresolved-note', 'Possible second profile needs review',
    '--actor-id', 'analyst:identity',
    '--review-db-path', str(review_db_path),
    '--output', 'json',
])
assert unresolved_exit == 0, unresolved_stderr
unresolved_reference = json.loads(unresolved_stdout)

report_exit, report_stdout, report_stderr = run_cli([
    'export-identity-report',
    '--review-db-path', str(review_db_path),
    '--output', 'json',
])
assert report_exit == 0, report_stderr
identity_report = json.loads(report_stdout)
assert identity_report['summary']['total_identities'] == 1
assert identity_report['summary']['total_memberships'] == 2
assert identity_report['summary']['external_reference_status_counts'] == {'attached': 1, 'unresolved': 1}
identity_report

{'identity_bundles': [{'external_references': [{'attached_at': '2026-03-18T19:40:48.492926+00:00',
     'attached_by': 'analyst:identity',
     'external_id': 'Q5388397',
     'identity_id': 'gid_d9199a02acfe3e5ab6c8a0b1',
     'provider': 'wikidata',
     'reference_id': 'gref_e917ba50d4ec48e390d0056b',
     'reference_label': 'Eric Olson',
     'reference_status': 'attached',
     'unresolved_note': None},
    {'attached_at': '2026-03-18T19:40:48.500746+00:00',
     'attached_by': 'analyst:identity',
     'external_id': None,
     'identity_id': 'gid_d9199a02acfe3e5ab6c8a0b1',
     'provider': 'wikidata',
     'reference_id': 'gref_0b276a3076854c008bd38f2a',
     'reference_label': None,
     'reference_status': 'unresolved',
     'unresolved_note': 'Possible second profile needs review'}],
   'identity': {'created_at': '2026-03-18T19:40:48.468553+00:00',
    'created_by': 'analyst:identity',
    'display_label': 'Eric Olson',
    'identity_id': 'gid_d9199a02acfe3e5ab6c8a0b1',
    'i

In [5]:
workspace.cleanup()
'cleanup_complete'

'cleanup_complete'